In [1]:
import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

import kagglehub


/kaggle/input/datasets/jvkrishwanth/mwdataset/sub-02-20260605T131514Z-3-001/sub-02/sub-02_sessions.tsv
/kaggle/input/datasets/jvkrishwanth/mwdataset/sub-02-20260605T131514Z-3-001/sub-02/sub-02_events.tsv
/kaggle/input/datasets/jvkrishwanth/mwdataset/sub-02-20260605T131514Z-3-001/sub-02/eeg/sub-02_ses-7_task-BreathCounting_eeg.json
/kaggle/input/datasets/jvkrishwanth/mwdataset/sub-02-20260605T131514Z-3-001/sub-02/eeg/sub-02_ses-8_task-BreathCounting_channels.tsv
/kaggle/input/datasets/jvkrishwanth/mwdataset/sub-02-20260605T131514Z-3-001/sub-02/eeg/sub-02_ses-5_task-BreathCounting_electrodes.tsv
/kaggle/input/datasets/jvkrishwanth/mwdataset/sub-02-20260605T131514Z-3-001/sub-02/eeg/sub-02_ses-10_task-BreathCounting_channels.tsv
/kaggle/input/datasets/jvkrishwanth/mwdataset/sub-02-20260605T131514Z-3-001/sub-02/eeg/sub-02_ses-8_task-BreathCounting_eeg.json
/kaggle/input/datasets/jvkrishwanth/mwdataset/sub-02-20260605T131514Z-3-001/sub-02/eeg/sub-02_ses-4_task-BreathCounting_eeg.json
/kaggle

In [2]:
"""Reproduce the reference band-power trajectories for MWDataset's 64 EEG channels.

The analysis uses all 64 BIDS EEG channels and removes ICA components correlated
with the available EXG channels before deriving Focus and MW epochs from the
MWDataset event codes.
"""

import gc
import os
from pathlib import Path
import warnings

# Avoid an MNE/Numba cache issue in the desktop Python installation.
os.environ.setdefault("NUMBA_DISABLE_JIT", "1")
os.environ.setdefault("MNE_NUM_JOBS", "1")

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import mne
import numpy as np
import pandas as pd
import seaborn as sns
from scipy.signal import welch
from scipy.stats import norm
import statsmodels.formula.api as smf


DATASET_ROOT = Path(os.environ.get("MWDATASET_ROOT", "/kaggle/input/datasets/jvkrishwanth/mwdataset"))
OUTPUT_DIR = Path("output") if Path("output").exists() else Path(os.environ.get("BAND_POWER_OUTPUT_DIR", "/kaggle/working/mwdataset_64ch_all_band_results"))
CSV_DIR = OUTPUT_DIR / "csv files" if (OUTPUT_DIR / "csv files").exists() else OUTPUT_DIR
PLOTS_DIR = OUTPUT_DIR / "plots" if (OUTPUT_DIR / "plots").exists() else OUTPUT_DIR

SUBJECTS = ("sub-01", "sub-02")
SESSIONS = range(1, 12)
BDF_ARCHIVES = {
    "sub-01": {1: "sub-01-20260605T131512Z-3-002", 2: "sub-01-20260605T131512Z-3-003", 3: "sub-01-20260605T131512Z-3-002", 4: "sub-01-20260605T131512Z-3-002", 5: "sub-01-20260605T131512Z-3-002", 6: "sub-01-20260605T131512Z-3-001", 7: "sub-01-20260605T131512Z-3-001", 8: "sub-01-20260605T131512Z-3-001", 9: "sub-01-20260605T131512Z-3-001", 10: "sub-01-20260605T131512Z-3-003", 11: "sub-01-20260605T131512Z-3-003"},
    "sub-02": {1: "sub-02-20260605T131514Z-3-003", 2: "sub-02-20260605T131514Z-3-002", 3: "sub-02-20260605T131514Z-3-002", 4: "sub-02-20260605T131514Z-3-001", 5: "sub-02-20260605T131514Z-3-001", 6: "sub-02-20260605T131514Z-3-002", 7: "sub-02-20260605T131514Z-3-002", 8: "sub-02-20260605T131514Z-3-001", 9: "sub-02-20260605T131514Z-3-001", 10: "sub-02-20260605T131514Z-3-001", 11: "sub-02-20260605T131514Z-3-003"},
}
METADATA_ARCHIVES = {"sub-01": "sub-01-20260605T131512Z-3-001", "sub-02": "sub-02-20260605T131514Z-3-001"}

BANDS = {"Delta": (1.0, 4.0), "Theta": (4.0, 8.0), "Alpha": (8.0, 12.0), "Beta": (12.0, 30.0), "Gamma": (30.0, 45.0)}
TARGET_SFREQ = 256.0
EPOCH_SECONDS = 5.0
WINDOW_SECONDS, STEP_SECONDS = 1.0, 0.5
EVENT_TRIAL_START, EVENT_MW_REPORT, EVENT_START_COUNTING = 10, 30, 50
FOCUS_START_OFFSET, MW_START_OFFSET = 1.0, -5.0


def session_paths(subject, session):
    label = f"ses-{session}"
    return (
        DATASET_ROOT / BDF_ARCHIVES[subject][session] / subject / "eeg" / f"{subject}_{label}_task-BreathCounting_eeg.bdf",
        DATASET_ROOT / METADATA_ARCHIVES[subject] / subject / "eeg" / f"{subject}_{label}_task-BreathCounting_channels.tsv",
    )


def deduplicate_events(events):
    if not len(events):
        return events
    frame = pd.DataFrame(events, columns=["sample", "previous", "code"])
    return frame.drop_duplicates(["sample", "code"]).sort_values(["sample", "code"])[["sample", "previous", "code"]].to_numpy(dtype=int)


def make_epoch_events(events, sfreq, last_sample, epoch_samples):
    rows = []
    for sample, _, code in events:
        if code == EVENT_MW_REPORT:
            start, state = sample + round(MW_START_OFFSET * sfreq), 1
        elif code in (EVENT_TRIAL_START, EVENT_START_COUNTING):
            start, state = sample + round(FOCUS_START_OFFSET * sfreq), 0
        else:
            continue
        if 0 <= start and start + epoch_samples <= last_sample:
            rows.append([start, 0, state])
    return deduplicate_events(np.asarray(rows, dtype=int)) if rows else np.empty((0, 3), dtype=int)


def ica_clean(raw, eeg_names, exg_names, correlation_threshold=0.30):
    """Remove ICA components correlated with EXG channels without dropping trials."""
    cleaned = raw.copy()
    if not exg_names:
        return cleaned.pick(eeg_names)
    ica = mne.preprocessing.ICA(
        n_components=0.99, method="fastica", random_state=42, max_iter="auto",
    )
    ica.fit(cleaned, picks=eeg_names, verbose=False)
    scores = [
        np.asarray(ica.score_sources(cleaned, target=cleaned.get_data(picks=[name])[0]), dtype=float)
        for name in exg_names
    ]
    excluded = np.flatnonzero(np.max(np.abs(np.vstack(scores)), axis=0) >= correlation_threshold)
    ica.apply(cleaned, exclude=excluded, verbose=False)
    return cleaned.pick(eeg_names)


def extract_windows():
    existing_csv = CSV_DIR / "all_band_window_level_data.csv"
    if existing_csv.exists():
        print(f"Reading existing window data from {existing_csv}")
        df = pd.read_csv(existing_csv)
        # Omit window 8 for Focus so both MW and Focus have exactly 8 windows (indices 0..7)
        df = df[df["Window_Index"] < 8].copy()
        return df
    if not DATASET_ROOT.exists():
        raise RuntimeError("Neither DATASET_ROOT nor existing CSV was found.")

    rows = []
    for subject in SUBJECTS:
        for session in SESSIONS:
            bdf_path, channels_path = session_paths(subject, session)
            print(f"Reading {subject}, session {session}", flush=True)
            raw = mne.io.read_raw_bdf(bdf_path, preload=False, stim_channel="Status", verbose=False)
            events = mne.find_events(raw, stim_channel="Status", shortest_event=1, consecutive=True, verbose=False)
            channel_table = pd.read_csv(channels_path, sep="	")
            eeg_names = channel_table.loc[channel_table["channelTypes"].fillna("").str.upper().eq("EEG"), "name"].astype(str).tolist()
            eeg_names = [name for name in eeg_names if name in raw.ch_names]
            if len(eeg_names) != 64:
                raise RuntimeError(f"{subject} session {session}: expected 64 EEG channels, found {len(eeg_names)}.")
            exg_names = [f"EXG{i}" for i in range(1, 9) if f"EXG{i}" in raw.ch_names]
            original_sfreq = raw.info["sfreq"]
            raw.pick(eeg_names + exg_names).load_data()
            raw.resample(TARGET_SFREQ, npad="auto", verbose=False)
            events[:, 0] = np.rint(events[:, 0] * TARGET_SFREQ / original_sfreq).astype(int)
            events = deduplicate_events(events)
            raw.filter(0.5, 45.0, fir_design="firwin", verbose=False)
            raw.set_channel_types({ch: "misc" for ch in exg_names if ch in raw.ch_names}, verbose=False)
            raw.set_eeg_reference("average", projection=False, verbose=False)
            raw = ica_clean(raw, eeg_names, exg_names)
            epoch_samples = round(EPOCH_SECONDS * raw.info["sfreq"])
            epoch_events = make_epoch_events(events, raw.info["sfreq"], raw.n_times - 1, epoch_samples)
            epochs = mne.Epochs(raw, epoch_events, event_id={"Focus": 0, "MW": 1}, tmin=0, tmax=EPOCH_SECONDS - 1 / raw.info["sfreq"], baseline=None, reject=None, flat=None, preload=True, event_repeated="drop", verbose=False)
            window_n, step_n = round(WINDOW_SECONDS * raw.info["sfreq"]), round(STEP_SECONDS * raw.info["sfreq"])
            for epoch_index, (epoch, event) in enumerate(zip(epochs.get_data(), epochs.events)):
                state = "MW" if event[2] == 1 else "Focus"
                for window_index, start in enumerate(range(0, epoch.shape[-1] - window_n + 1, step_n)):
                    # Omit the last window (window 8) for BOTH MW and Focus so lengths are identical
                    if start + window_n > round(4.9 * raw.info["sfreq"]):
                        continue
                    freqs, psd = welch(epoch[:, start:start + window_n], fs=raw.info["sfreq"], nperseg=window_n, axis=-1)
                    row = {"Subject": subject, "Session": session, "Epoch": f"{subject}_{session}_{epoch_index}", "State": state, "Epoch_Index": epoch_index, "Window_Index": window_index, "Time_s": (start + window_n / 2) / raw.info["sfreq"], "N_EEG_Channels": len(eeg_names)}
                    for band, (low, high) in BANDS.items():
                        mask = (freqs >= low) & (freqs < high)
                        # Mean is explicitly across frequency bins and all 64 EEG channels.
                        row[f"Log10_{band}_Power"] = np.log10(psd[:, mask].mean() + 1e-20)
                    rows.append(row)
            del epochs, raw
            gc.collect()
    if not rows:
        raise RuntimeError("No valid MWDataset epochs were extracted.")
    return pd.DataFrame(rows)


def epoch_summaries(windows, band):
    value = f"Log10_{band}_Power"
    rows = []
    for (subject, epoch, state), part in windows.groupby(["Subject", "Epoch", "State"], observed=True):
        row = {"Subject": subject, "Epoch": epoch, "State": state, "Mean_log10_power": part[value].mean(), "N_Windows": len(part)}
        row["Slope_log10_power_per_second"] = np.polyfit(part.Time_s, part[value], 1)[0] if len(part) >= 3 else np.nan
        rows.append(row)
    return pd.DataFrame(rows)


def contrast_result(fit, contrast, band, analysis):
    params, cov = fit.params, fit.cov_params().loc[fit.params.index, fit.params.index]
    estimate = float(contrast @ params.to_numpy()); se = float(np.sqrt(contrast @ cov.to_numpy() @ contrast)); z = estimate / se
    p_value = 2 * norm.sf(abs(z))
    return {"Band": band, "Analysis": analysis, "Estimate": estimate, "SE": se, "z": z, "CI_95_low": estimate - 1.96 * se, "CI_95_high": estimate + 1.96 * se, "p_value": p_value}


def analyse_band(windows, band):
    summary = epoch_summaries(windows, band); summary["State"] = pd.Categorical(summary.State, categories=["Focus", "MW"])
    mean_fit = smf.ols("Mean_log10_power ~ C(State, Treatment(reference='Focus'))", summary).fit(cov_type="cluster", cov_kwds={"groups": summary.Subject})
    mean_names = list(mean_fit.params.index); state_name = next(n for n in mean_names if n.startswith("C(State,")); mean_contrast = np.zeros(len(mean_names)); mean_contrast[mean_names.index(state_name)] = 1
    trend = summary.dropna(subset=["Slope_log10_power_per_second"])
    trend_fit = smf.ols("Slope_log10_power_per_second ~ C(State, Treatment(reference='Focus'))", trend).fit(cov_type="cluster", cov_kwds={"groups": trend.Subject})
    trend_names = list(trend_fit.params.index); trend_state = next(n for n in trend_names if n.startswith("C(State,")); focus = np.zeros(len(trend_names)); focus[trend_names.index("Intercept")] = 1; mw = focus.copy(); mw[trend_names.index(trend_state)] = 1
    return summary, [contrast_result(mean_fit, mean_contrast, band, "MW minus Focus mean power"), contrast_result(trend_fit, focus, band, "Focus change over time"), contrast_result(trend_fit, mw, band, "MW change over time")]


def plot_band(windows, band):
    value = f"Log10_{band}_Power"
    subject_curve = windows.groupby(["Subject", "State", "Time_s"], as_index=False)[value].mean()
    stats = subject_curve.groupby(["State", "Time_s"], as_index=False)[value].agg(mean="mean", sem="sem")

    fig, ax_mw = plt.subplots(figsize=(12, 7.8))
    ax_focus = ax_mw.twiny()

    # MW series on primary (bottom) axis
    s_mw = stats[stats.State == "MW"]
    time_mw = s_mw.Time_s + MW_START_OFFSET
    l_mw = ax_mw.plot(time_mw, s_mw["mean"], marker="o", lw=3.8, markersize=10, label="MW", color="#D1495B")
    ax_mw.fill_between(time_mw, s_mw["mean"] - s_mw["sem"], s_mw["mean"] + s_mw["sem"], color="#D1495B", alpha=0.2)

    # Focus series on secondary (top) axis
    s_focus = stats[stats.State == "Focus"]
    time_focus = s_focus.Time_s + FOCUS_START_OFFSET
    l_focus = ax_focus.plot(time_focus, s_focus["mean"], marker="o", lw=3.8, markersize=10, label="Focus", color="#2878B5")
    ax_focus.fill_between(time_focus, s_focus["mean"] - s_focus["sem"], s_focus["mean"] + s_focus["sem"], color="#2878B5", alpha=0.2)

    # Align windows 0..7 perfectly: MW [-4.5, -1.0] corresponds to Focus [1.5, 5.0]
    ax_mw.set_xlim(-4.8, -0.7)
    ax_focus.set_xlim(1.2, 5.3)

    ax_mw.set_xticks([-4.5, -3.5, -2.5, -1.5, -1.0])
    ax_focus.set_xticks([1.5, 2.5, 3.5, 4.5, 5.0])

    band_ylims = {
        "Delta": (-11.55, -11.10),
        "Theta": (-12.15, -11.85),
        "Alpha": (-12.35, -11.45),
        "Beta":  (-12.70, -12.30),
        "Gamma": (-13.35, -12.85),
    }
    band_yticks = {
        "Delta": [-11.5, -11.4, -11.3, -11.2, -11.1],
        "Theta": [-12.1, -12.0, -11.9],
        "Alpha": [-12.3, -12.1, -11.9, -11.7, -11.5],
        "Beta":  [-12.7, -12.6, -12.5, -12.4, -12.3],
        "Gamma": [-13.3, -13.2, -13.1, -13.0, -12.9],
    }
    if band in band_ylims:
        ax_mw.set_ylim(band_ylims[band])
        ax_mw.set_yticks(band_yticks[band])

    ax_mw.set_xlabel("Time relative to button press (s) [MW]", color="#D1495B", fontsize=22, fontweight="bold", labelpad=14)
    ax_focus.set_xlabel("Time relative to trial start (s) [Focus]", color="#2878B5", fontsize=22, fontweight="bold", labelpad=14)
    ax_mw.set_ylabel(f"Mean log10 {band.lower()} PSD (power/Hz)", fontsize=24, fontweight="bold", labelpad=14)

    ax_mw.tick_params(axis="x", colors="#D1495B", labelsize=20, width=1.5, length=6)
    ax_focus.tick_params(axis="x", colors="#2878B5", labelsize=20, width=1.5, length=6)
    ax_mw.tick_params(axis="y", labelsize=20, width=1.5, length=6)

    for label in ax_mw.get_xticklabels() + ax_focus.get_xticklabels() + ax_mw.get_yticklabels():
        label.set_fontweight("bold")

    ax_focus.grid(False)
    ax_mw.grid(True, color="#c7c7c7", linewidth=1.2)
    for spine in ax_mw.spines.values():
        spine.set_edgecolor("#c7c7c7")
        spine.set_linewidth(1.2)

    plt.title(f"{band}-band power over time", y=1.22, fontsize=28, fontweight="bold")

    # Combined legend: Focus first, then MW
    lines = l_focus + l_mw
    labels = [l.get_label() for l in lines]
    leg = ax_focus.legend(lines, labels, title="State", bbox_to_anchor=(1.04, 1), loc="upper left", fontsize=22, title_fontsize=24)
    leg.get_title().set_fontweight("bold")
    for t in leg.get_texts():
        t.set_fontweight("bold")

    fig.savefig(PLOTS_DIR / f"{band.lower()}_band_power_over_time.png", dpi=200, bbox_inches="tight")
    plt.close(fig)
    return subject_curve


def plot_all_bands(curves):
    fig, axes = plt.subplots(1, 2, figsize=(18, 7.5), sharey=True)
    colours = {"Delta": "#4C78A8", "Theta": "#F58518", "Alpha": "#54A24B", "Beta": "#E45756", "Gamma": "#B279A2"}
    for ax, state in zip(axes, ("Focus", "MW")):
        for band, curve in curves.items():
            value = f"Log10_{band}_Power"; part = curve[curve.State == state]; z = (part[value] - curve[value].mean()) / curve[value].std(ddof=0); summary = pd.DataFrame({"Time_s": part.Time_s, "z": z}).groupby("Time_s").z.agg(["mean", "sem"]).reset_index()
            plot_time = summary.Time_s + MW_START_OFFSET if state == "MW" else summary.Time_s + FOCUS_START_OFFSET
            ax.plot(plot_time, summary["mean"], marker="o", lw=3.5, markersize=9, label=band, color=colours[band])
            ax.fill_between(plot_time, summary["mean"] - summary["sem"], summary["mean"] + summary["sem"], color=colours[band], alpha=0.18)
        ax.axhline(0, color="black", lw=1.2, ls="--", alpha=0.7)
        ax.set_title(state, fontsize=28, fontweight="bold", pad=16)
        ax.set_xlabel("Time relative to button press (s)" if state == "MW" else "Time relative to trial start (s)", fontsize=24, fontweight="bold", labelpad=12)
        ax.set_xticks([-4, -3, -2, -1] if state == "MW" else [2, 3, 4, 5])
        ax.tick_params(axis="both", labelsize=22, width=1.5, length=6)
        for label in ax.get_xticklabels() + ax.get_yticklabels():
            label.set_fontweight("bold")
        ax.grid(True, color="#c7c7c7", linewidth=1.2)
        for spine in ax.spines.values():
            spine.set_edgecolor("#c7c7c7")
            spine.set_linewidth(1.2)
    axes[0].set_ylim(-1.2, 1.2)
    axes[0].set_yticks([-1.0, -0.5, 0.0, 0.5, 1.0])
    axes[0].set_ylabel("Standardised mean log10 PSD (z)", fontsize=24, fontweight="bold", labelpad=12)
    for label in axes[0].get_yticklabels():
        label.set_fontweight("bold")
    leg = axes[1].legend(title="Band", bbox_to_anchor=(1.02, 1), loc="upper left", fontsize=22, title_fontsize=24)
    leg.get_title().set_fontweight("bold")
    for t in leg.get_texts():
        t.set_fontweight("bold")
    fig.suptitle("All EEG bands: power trajectories", y=1.03, fontsize=30, fontweight="bold")
    fig.tight_layout()
    fig.savefig(PLOTS_DIR / "all_bands_power_over_time_standardised.png", dpi=200, bbox_inches="tight")
    plt.close(fig)


def main():
    warnings.filterwarnings("ignore"); mne.set_log_level("WARNING"); sns.set_theme(style="whitegrid", context="talk"); CSV_DIR.mkdir(parents=True, exist_ok=True); PLOTS_DIR.mkdir(parents=True, exist_ok=True)
    windows = extract_windows(); windows.to_csv(CSV_DIR / "all_band_window_level_data.csv", index=False)
    all_results, curves = [], {}
    for band in BANDS:
        summary, results = analyse_band(windows, band); summary.to_csv(CSV_DIR / f"{band.lower()}_epoch_summaries.csv", index=False); all_results.extend(results); curves[band] = plot_band(windows, band)
        results = pd.DataFrame(all_results)
    results["Statistical_decision_alpha_0.05"] = np.where(
        results["p_value"] < 0.05, "Statistically significant", "Not statistically significant"
    )
    results.to_csv(CSV_DIR / "all_band_statistical_results.csv", index=False)
    plot_all_bands(curves)
    print("\nStatistical results:\n", results.to_string(index=False))
    significant_results = results.loc[results["p_value"] < 0.05].sort_values("p_value").copy()
    print("\nAll statistically significant results (p < 0.05):\n", significant_results.to_string(index=False))
    remaining_results = results.loc[results["p_value"] >= 0.05].nsmallest(5, "p_value")
    print("\nFive lowest p-value results among the remaining analyses:\n", remaining_results.to_string(index=False))
    print(f"\nSaved 64-channel plots to {PLOTS_DIR} and CSVs to {CSV_DIR}")

if __name__ == "__main__":
    main()

Reading existing window data from output\csv files\all_band_window_level_data.csv

Statistical results:
  Band                  Analysis  Estimate       SE          z  CI_95_low  CI_95_high       p_value Statistical_decision_alpha_0.05
Delta MW minus Focus mean power  0.083015 0.055241   1.502783  -0.025257    0.191287  1.328950e-01   Not statistically significant
Delta    Focus change over time -0.009337 0.006089  -1.533364  -0.021272    0.002598  1.251861e-01   Not statistically significant
Delta       MW change over time  0.018215 0.005001   3.642174   0.008413    0.028018  2.703451e-04       Statistically significant
Theta MW minus Focus mean power  0.033344 0.005853   5.697020   0.021872    0.044815  1.219193e-08       Statistically significant
Theta    Focus change over time -0.003804 0.007628  -0.498662  -0.018755    0.011147  6.180173e-01   Not statistically significant
Theta       MW change over time -0.001909 0.003235  -0.590019  -0.008249    0.004432  5.551781e-01   Not stat